In [ ]:
import math,torch,torch.nn as nn,torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets,transforms
from torchvision.utils import save_image

device = "cuda" if torch.cuda.is_available() else "cpu"
T,BATCH,EPOCHS = 400,128,20

betas = torch.linspace(1e-4,0.02,T,device=device)
alphas = 1. - betas
acp = torch.cumprod(alphas,0)
sqrt_acp,sqrt_om = acp.sqrt(),(1. - acp).sqrt()
coef1,coef2 = (1. / alphas).sqrt(),betas / sqrt_om

def extract(a,t):
    return a[t].view(-1,1,1,1)

class SinPosEmb(nn.Module):
    def __init__(self,dim):
        super().__init__()
        self.dim = dim
    def forward(self,t):
        half = self.dim // 2
        freq = torch.exp(torch.arange(half,device=t.device) * -(math.log(1e4)/(half-1)))
        emb = t[:,None].float() * freq
        return torch.cat((emb.sin(),emb.cos()),dim=-1)

class Block(nn.Module):
    def __init__(self,in_ch,out_ch,tdim):
        super().__init__()
        self.c1,self.c2 = nn.Conv2d(in_ch,out_ch,3,padding=1),nn.Conv2d(out_ch,out_ch,3,padding=1)
        self.n1,self.n2 = nn.GroupNorm(8,in_ch),nn.GroupNorm(8,out_ch)
        self.tmlp = nn.Linear(tdim,out_ch)
        self.skip = nn.Identity() if in_ch == out_ch else nn.Conv2d(in_ch,out_ch,1)
    def forward(self,x,t):
        h = self.c1(F.silu(self.n1(x)))
        h = h + self.tmlp(t)[:,:,None,None]
        h = self.c2(F.silu(self.n2(h)))
        return h + self.skip(x)

class Attn(nn.Module):
    def __init__(self,ch):
        super().__init__()
        self.norm,self.qkv,self.proj = nn.GroupNorm(8,ch),nn.Conv2d(ch,3 * ch,1),nn.Conv2d(ch,ch,1)
    def forward(self,x):
        b,c,h,w = x.shape
        q,k,v = (m.flatten(2).transpose(1,2) for m in self.qkv(self.norm(x)).chunk(3,1))
        o = F.softmax(q @ k.transpose(-2,-1) / math.sqrt(c),dim=-1) @ v
        return x + self.proj(o.transpose(1,2).reshape(b,c,h,w))

class UNet(nn.Module):
    def __init__(self,ch=64,tdim=128):
       super().__init__()
       self.sin = SinPosEmb(tdim)
       self.tmlp = nn.Sequential(nn.Linear(tdim,tdim),nn.SiLU(),nn.Linear(tdim,tdim))
       self.inp = nn.Conv2d(1,ch,3,padding=1)
       self.d1,self.dn1 = Block(ch,ch,tdim),nn.Conv2d(ch,2 * ch,3,2,1)
       self.d2,self.dn2 = Block(2 * ch,2 * ch,tdim),nn.Conv2d(2 * ch,2 * ch,3,2,1)
       self.mid,self.attn = Block(2 * ch,2 * ch,tdim),Attn(2 * ch)
       self.up2,self.u2 = nn.ConvTranspose2d(2 * ch,2 * ch,4,2,1),Block(2 * ch,2 * ch,tdim)
       self.up1,self.u1 = nn.ConvTranspose2d(2 * ch,ch,4,2,1),Block(ch,ch,tdim)
       self.out = nn.Conv2d(ch,1,3,padding=1)
    def forward(self,x,t):
        t = self.tmlp(self.sin(t))
        h1 = self.d1(self.inp(x),t)
        h2 = self.d2(self.dn1(h1),t)
        h = self.attn(self.mid(self.dn2(h2),t))
        h = self.u2(self.up2(h),t) + h2
        h = self.u1(self.up1(h),t) + h1
        return self.out(h)

model = UNet().to(device)
opt = torch.optim.AdamW(model.parameters(),lr=2e-4)
ds = datasets.MNIST("./data",train=True,download=False,
                     transform=transforms.Compose([transforms.ToTensor(),transforms.Normalize(0.5,0.5)]))
loader = DataLoader(ds,batch_size=BATCH,shuffle=True,drop_last=True)

def train():
    model.train()
    for ep in range(EPOCHS):
        for x,_ in loader:
            x = x.to(device)
            t = torch.randint(0,T,(x.size(0),),device=device)
            noise = torch.randn_like(x)
            x_t = extract(sqrt_acp,t) * x + extract(sqrt_om,t) * noise
            loss = F.mse_loss(model(x_t,t),noise)
            opt.zero_grad();loss.backward();opt.step()
        print(f"epoch {ep + 1}/{EPOCHS} loss {loss.item():.4f}")
@torch.no_grad()
def sample(model,n=64):
    model.eval()
    x = torch.randn(n,1,28,28,device=device)
    for t in reversed(range(T)):
        tb = torch.full((n,),t,device=device,dtype=torch.long)
        eps = model(x,tb)
        mean = extract(coef1,tb) * (x - extract(coef2,tb) * eps)
        x = mean + betas[t].sqrt() * torch.randn_like(x) if t > 0 else mean
    return x
if __name__ == "__main__":
    train()
    save_image(sample(model),"ddpm_sample.png",nrow=8,normalize=True,value_range=(-1,1))

In [11]:
import numpy as np
import pandas as pd
df = pd.DataFrame({'Class':[1,1,2,2],
                   'Name':['San Zhang','San Zhang','Si Li','Si Li'],
                   'Subject':['Chinese','Math','Chinese','Math'],
                   'Grade':[80,75,90,85]})
print(df)
df.pivot(index='Name',columns='Subject',values='Grade')

   Class       Name  Subject  Grade
0      1  San Zhang  Chinese     80
1      1  San Zhang     Math     75
2      2      Si Li  Chinese     90
3      2      Si Li     Math     85


Subject,Chinese,Math
Name,,
San Zhang,80,75
Si Li,90,85


In [16]:
df = pd.DataFrame({'Name':['San Zhang','San Zhang',
                           'San Zhang','San Zhang',
                           'Si Li','Si Li','Si Li','Si Li'],
                   'Subject':['Chinese','Chinese','Math','Math',
                              'Chinese','Chinese','Math','Math'],
                    'Grade':[80,90,100,90,70,80,85,95]})
print(df)
print(df.pivot_table(index='Name',columns='Subject',values='Grade',aggfunc='mean'))
df.pivot_table(index='Name',columns='Subject',values='Grade',aggfunc=lambda x:x.mean())

        Name  Subject  Grade
0  San Zhang  Chinese     80
1  San Zhang  Chinese     90
2  San Zhang     Math    100
3  San Zhang     Math     90
4      Si Li  Chinese     70
5      Si Li  Chinese     80
6      Si Li     Math     85
7      Si Li     Math     95
Subject    Chinese  Math
Name                    
San Zhang     85.0  95.0
Si Li         75.0  90.0


Subject,Chinese,Math
Name,,
San Zhang,85.0,95.0
Si Li,75.0,90.0


In [ ]:
df = pd.DataFrame({'Class':[1,2],
                   'Name':['San Zhang','Si Li'],
                   'Chinese':[80,90],
                   'Math':[80,75]})
print(df)
df_melted = df.melt(id_vars=['Class','Name'],
                    value_vars=['Chinese','Math'],
                    var_name='Subject',
                    value_name='Grade')
print(df_melted)

   Class       Name  Chinese  Math
0      1  San Zhang       80    80
1      2      Si Li       90    75
   Class       Name  Subject  Grade
0      1  San Zhang  Chinese     80
1      2      Si Li  Chinese     90
2      1  San Zhang     Math     80
3      2      Si Li     Math     75


In [23]:
df_unmelted = df_melted.pivot(index=['Class','Name'],columns='Subject',values='Grade')
print(df_unmelted)
df_unmelted = df_unmelted.reset_index().rename_axis(columns={'Subject':''})
df_unmelted.equals(df)

Subject          Chinese  Math
Class Name                    
1     San Zhang       80    80
2     Si Li           90    75


True

In [29]:
import numpy as np
import pandas as pd
df = pd.read_csv('data1/learn_pandas.csv',usecols=['Grade','Name','Gender','Height','Weight','Transfer'])
print(df.isna().head())
print(df.isna().sum()/df.shape[0])
print(df[df.Height.isna()].head())

   Grade   Name  Gender  Height  Weight  Transfer
0  False  False   False   False   False     False
1  False  False   False   False   False     False
2  False  False   False   False   False     False
3  False  False   False    True   False     False
4  False  False   False   False   False     False
Grade       0.000
Name        0.000
Gender      0.000
Height      0.085
Weight      0.055
Transfer    0.060
dtype: float64
        Grade          Name  Gender  Height  Weight Transfer
3   Sophomore  Xiaojuan Sun  Female     NaN    41.0        N
12     Senior      Peng You  Female     NaN    48.0      NaN
26     Junior     Yanli You  Female     NaN    48.0        N
36   Freshman  Xiaojuan Qin    Male     NaN    79.0        Y
60   Freshman    Yanpeng Lv    Male     NaN    65.0        N


In [34]:
sub_set = df[['Height','Weight','Transfer']]
print(df[sub_set.isna().all(axis=1)])
print(df[sub_set.isna().any(axis=1)].head())
df[sub_set.notna().all(axis=1)].head()

      Grade          Name Gender  Height  Weight Transfer
102  Junior  Chengli Zhao   Male     NaN     NaN      NaN
        Grade           Name  Gender  Height  Weight Transfer
3   Sophomore   Xiaojuan Sun  Female     NaN    41.0        N
9      Junior        Juan Xu  Female   164.8     NaN        N
12     Senior       Peng You  Female     NaN    48.0      NaN
21     Senior  Xiaopeng Shen    Male   166.0    62.0      NaN
26     Junior      Yanli You  Female     NaN    48.0        N


,Grade,Name,Gender,Height,Weight,Transfer
0,Freshman,Gaopeng Yang,Female,158.9,46.0,N
1,Freshman,Changqiang You,Male,166.5,70.0,N
2,Senior,Mei Sun,Male,188.9,89.0,N
4,Sophomore,Gaojuan You,Male,174.0,74.0,N
5,Freshman,Xiaoli Qian,Female,158.0,51.0,N


In [35]:
res = df.dropna(how='any',subset=['Height','Weight'])
res.shape

(174, 6)

In [38]:
res = df.dropna(axis=1,thresh=df.shape[0]-15)
res.head()

,Grade,Name,Gender,Weight,Transfer
0,Freshman,Gaopeng Yang,Female,46.0,N
1,Freshman,Changqiang You,Male,70.0,N
2,Senior,Mei Sun,Male,89.0,N
3,Sophomore,Xiaojuan Sun,Female,41.0,N
4,Sophomore,Gaojuan You,Male,74.0,N


In [40]:
res = df.loc[df[['Height','Weight']].notna().all(axis=1)]
print(res.shape)
res = df.loc[:,-(df.isna().sum()>15)]
print(res.head())

(174, 6)
       Grade            Name  Gender  Weight Transfer
0   Freshman    Gaopeng Yang  Female    46.0        N
1   Freshman  Changqiang You    Male    70.0        N
2     Senior         Mei Sun    Male    89.0        N
3  Sophomore    Xiaojuan Sun  Female    41.0        N
4  Sophomore     Gaojuan You    Male    74.0        N
